# Explorations : benchmarks historiques, marché et extensions OTC

Ce notebook conserve les expériences utiles à la compréhension du problème mais
qui ne sélectionnent pas la politique conditionnelle finale : surface de prix,
payoff moyen, contrôle Pathwise payoff non conditionnel, scénarios OTC, coûts
fondés sur des cotations et prolongements non encore implémentés.

Les trois premières parties sont autonomes. Les analyses dépendant d'une politique
conditionnelle déjà calibrée sont désactivées par défaut et exigent les objets du
notebook principal dans le noyau courant. Elles ne sont jamais exécutées ni exportées
silencieusement par le parcours principal.

Les anciens exports restent conservés dans `output/`. Les cellules déplacées sont
livrées sans sorties d'exécution pour éviter de présenter des résultats historiques
comme s'ils venaient de cette nouvelle exécution.


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys
from typing import Any, Dict, Optional
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from IPython.display import display

# Horodatage de référence de cette exécution complète. Il est réinitialisé
# au lancement de la première cellule de code lors d'un Run All.
NOTEBOOK_RUN_STARTED_AT = datetime.now().astimezone()

# Rend le package src/ importable quel que soit le noyau Jupyter choisi.
_search_bases = (
    Path.cwd(),
    Path.cwd() / "autocall-replication",
    *(candidate for parent in Path.cwd().parents
      for candidate in (parent, parent / "autocall-replication")),
)
_project_root = next(
    (candidate for candidate in _search_bases
     if (candidate / "pyproject.toml").is_file()
     and (candidate / "src" / "autocall_replication").is_dir()),
    None,
)
if _project_root is None:
    raise RuntimeError(
        "Projet autocall-replication introuvable depuis le dossier de travail Jupyter."
    )
_src_path = str(_project_root / "src")
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)

# Recharge automatiquement les modules locaux modifiés pendant une session.
# Cela évite qu'un noyau déjà ouvert conserve une ancienne interface de solveur.
try:
    _ipython = get_ipython()
except NameError:
    _ipython = None
if _ipython is not None:
    _ipython.run_line_magic("load_ext", "autoreload")
    _ipython.run_line_magic("autoreload", "2")

try:
    import yfinance as yf
except ImportError:
    yf = None


In [ ]:
RUN_MODE = "fast"  # "report" augmente uniquement le volume numérique.
SHOW_TECHNICAL_DETAILS = False
RUN_OTC_FUNDING_EXPLORATION = False
RUN_POLICY_MARKET_COSTS = False
RUN_OPTIONAL_MARKET_EXPLORATION = False
RUN_FIGURES = {}
EXECUTION_CONFIGS = {
    "fast": {"pricing_paths": 2_500, "surface_paths": 250,
             "benchmark_paths": 250, "alpha_paths": 200, "pathwise_paths": 375,
             "funding_paths": 250},
    "report": {"pricing_paths": 20_000, "surface_paths": 2_000,
               "benchmark_paths": 2_000, "alpha_paths": 2_000, "pathwise_paths": 2_000,
               "funding_paths": 2_000},
}
if RUN_MODE not in EXECUTION_CONFIGS:
    raise ValueError("RUN_MODE doit être 'fast' ou 'report'.")
CFG = EXECUTION_CONFIGS[RUN_MODE]
MAIN_FAMILIES = ("calls_puts", "full")
FAMILY_LABELS = {"calls_puts": "Calls + Puts", "full": "Full (avec binaires)"}
SOLVER = "legacy"
display(pd.Series({"mode": RUN_MODE, **CFG}, name="configuration exploratoire"))


## 1. Contrat cible et hypothèses

### 1.1 Définition du produit

On fixe ici une seule fois les caractéristiques contractuelles et les hypothèses Black-Scholes. Toutes les sections suivantes utilisent ces mêmes objets afin d’éviter les incohérences de dates, de nominal ou de paramètres de marché.


In [ ]:
from autocall_replication.products import (
    AutocallProduct,
    VanillaProduct,
    EuropeanCall,
    EuropeanPut,
    BinaryCall,
    BinaryPut,
    Cash,
    build_vanilla_universe,
    vanilla_universe_to_dataframe,
    norm_cdf,
    norm_pdf,
    bs_d1_d2,
    _year_fraction,
)


In [ ]:
# Produit cible et hypothèses de marché communes à toutes les sections.
prod = AutocallProduct(
    notional=100.0,
    autocall_strike=1.00,
    coupon_rate=0.02,
    down_barrier=0.70,
    observation_months=[3, 6, 9, 12],
)

valuation_date = pd.Timestamp("2026-06-20")
contract_start_date = pd.offsets.BDay().rollforward(valuation_date.normalize())
maturity_date = pd.offsets.BDay().rollforward(
    prod.build_observation_dates(contract_start_date)[-1].normalize()
)
spot0 = 100.0
rate = 0.03
dividend_yield = 0.01
vol = 0.20

contract_summary = pd.Series({
    "nominal": prod.notional,
    "spot initial": spot0,
    "seuil de rappel": prod.autocall_strike,
    "coupon par observation": prod.coupon_rate,
    "barrière de protection": prod.down_barrier,
    "observations (mois)": prod.observation_months,
    "date de valorisation": valuation_date.date(),
    "fixing effectif": contract_start_date.date(),
    "maturité effective": maturity_date.date(),
    "taux sans risque": rate,
    "rendement du dividende": dividend_yield,
    "volatilité": vol,
}, name="valeur")
display(contract_summary.to_frame())


## 2. Validation du pricing Monte-Carlo

### 2.1 Méthode

Le sous-jacent suit ici une dynamique Black-Scholes sous la mesure risque-neutre. Le prix de l’autocall est la moyenne des cashflows actualisés simulés. L’erreur standard mesure l’incertitude Monte-Carlo, pas le risque économique du produit.


In [ ]:
from autocall_replication.monte_carlo import simulate_gbm_path, price_autocall_bs_mc
from autocall_replication.replication.common import (
    benchmark_metrics,
    build_autocall_training_grid_axes,
    build_naive_vanilla_basis,
    plot_benchmark_surface,
)
from autocall_replication.replication.price_surface import (
    price_autocall_on_tensor_bs_mc,
    price_vanilla_basis_on_grid,
    run_naive_benchmark_price,
)
from autocall_replication.replication.portfolio_diagnostics import (
    format_grouped_portfolio_metrics,
    format_portfolio_metrics,
    portfolio_metrics_summary,
)


In [ ]:
pricing_result = price_autocall_bs_mc(
    product=prod,
    spot0=spot0,
    valuation_date=valuation_date,
    maturity_date=maturity_date,
    rate=rate,
    dividend_yield=dividend_yield,
    vol=vol,
    n_paths=CFG["pricing_paths"],
)

display(pd.Series({
    "prix actualisé": pricing_result["actualized_price"],
    "erreur standard": pricing_result["std_error"],
    "nombre de trajectoires": CFG["pricing_paths"],
}, name="pricing Monte-Carlo"))


### 2.2 Surface date–spot

La surface suivante valorise le même contrat pour plusieurs dates et niveaux de spot. Elle sert à vérifier la forme du prix et constitue une première cible pour les benchmarks statiques.

Les lignes sont les dates de valorisation, les colonnes les niveaux du sous-jacent. Chaque cellule contient une estimation Monte-Carlo du prix.


In [ ]:
dates, spots = build_autocall_training_grid_axes(
    spot0=spot0,
    valuation_date=valuation_date,
    maturity_date=maturity_date,
    n_spots=15,
    spot_min_mult=0.6,
    spot_max_mult=1.4,
    include_call_dates=True,
    product=prod,
)

price_tensor = price_autocall_on_tensor_bs_mc(
    product=prod,
    dates=dates,
    spots=spots,
    maturity_date=maturity_date,
    rate=rate,
    dividend_yield=dividend_yield,
    vol=vol,
    n_paths=CFG["surface_paths"],
    seed=42,
)

price_surface_table = pd.DataFrame(
    price_tensor,
    index=[date.strftime("%Y-%m-%d") for date in dates],
    columns=np.round(spots, 2),
)
price_surface_table.index.name = "date"
price_surface_table.columns.name = "spot"
display(price_surface_table.round(4))


#### Lecture visuelle de la surface

La couleur représente le prix estimé de l’autocall. Les deux lignes verticales repèrent la barrière de protection et le seuil de rappel, exprimés relativement au spot initial. Cette figure sert à identifier rapidement les zones où le prix change fortement ; le tableau précédent conserve les valeurs exactes.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.8))
heatmap = ax.imshow(
    price_surface_table.to_numpy(dtype=float),
    aspect="auto",
    origin="upper",
    cmap="viridis",
)

tick_step = max(1, len(spots) // 7)
x_positions = np.arange(len(spots))[::tick_step]
ax.set_xticks(x_positions)
ax.set_xticklabels([f"{spots[index]:.1f}" for index in x_positions])
ax.set_yticks(np.arange(len(dates)))
ax.set_yticklabels([date.strftime("%Y-%m-%d") for date in dates])
ax.set_xlabel("Niveau du sous-jacent")
ax.set_ylabel("Date de valorisation")
ax.set_title("Surface de prix de l’autocall — date × spot")

reference_levels = {
    "Barrière de protection": spot0 * prod.down_barrier,
    "Seuil de rappel": spot0 * prod.autocall_strike,
}
for label, level in reference_levels.items():
    nearest_index = int(np.argmin(np.abs(spots - level)))
    ax.axvline(nearest_index, linestyle="--", linewidth=1.5, label=f"{label} ({level:.1f})")

fig.colorbar(heatmap, ax=ax, label="Prix estimé")
ax.legend(loc="upper left", bbox_to_anchor=(1.12, 1.0))
plt.tight_layout()
RUN_FIGURES["autocall_price_surface"] = fig
plt.show()


## Recherche d'alpha propre aux deux objectifs exploratoires

Les alphas sont choisis sur une validation interne distincte de la calibration.
Ils ne sont ni réutilisés par la politique conditionnelle ni promus vers le
notebook principal. FAST et REPORT parcourent les mêmes valeurs d'alpha ;
seul le nombre de trajectoires change.


In [ ]:
from autocall_replication.replication.mean_payoff import run_naive_benchmark_payoff
from autocall_replication.replication.optimization import SolverConfig, solve_replication
from autocall_replication.replication.hyperparameter_study import (
    HyperparameterGrid, exploratory_alpha_values,
    plot_exploratory_alpha_grid, select_exploratory_benchmark_alphas,
)

GRID = HyperparameterGrid()
_benchmark_penalties = {
    "price_surface": ("l0", "l1", "l2", "elastic_net", "l1_2"),
    "mean_payoff": ("l2",),
}
_benchmark_functions = {
    "price_surface": run_naive_benchmark_price,
    "mean_payoff": run_naive_benchmark_payoff,
}
benchmark_alpha_rows = []
for benchmark, penalties in _benchmark_penalties.items():
    for family in MAIN_FAMILIES:
        common = dict(
            product=prod, spot0=spot0, valuation_date=valuation_date,
            maturity_date=maturity_date, rate=rate,
            dividend_yield=dividend_yield, vol=vol, family=family,
            n_spots=15, n_paths=CFG["alpha_paths"], seed=42,
        )
        baseline = _benchmark_functions[benchmark](
            **common, penalty="l2", alpha=1e-4, solver="legacy",
        )
        matrix = np.asarray(baseline["A"], dtype=float)
        target = np.asarray(baseline["target_tensor"], dtype=float).reshape(-1)
        split = int(0.7 * len(baseline["dates"])) * len(baseline["spots"])
        train_A, train_b = matrix[:split], target[:split]
        validation_A, validation_b = matrix[split:], target[split:]
        for penalty in penalties:
            for alpha in exploratory_alpha_values(GRID, penalty):
                fit = solve_replication(
                    train_A, train_b,
                    SolverConfig(
                        solver="proximal" if penalty == "l0" else "legacy",
                        penalty=penalty, alpha=float(alpha),
                    ),
                )
                benchmark_alpha_rows.append({
                    "benchmark": benchmark, "family": family,
                    "penalty": penalty, "alpha": float(alpha),
                    "converged": bool(fit.converged),
                    "validation_rmse": float(np.sqrt(np.mean(
                        (validation_b - validation_A @ fit.weights) ** 2
                    ))),
                    "n_options_active": int(np.count_nonzero(
                        np.abs(fit.weights[1:]) > 1e-6
                    )),
                })
    print(f"Recherche d'alpha terminée : {benchmark}")

benchmark_alpha_table = pd.DataFrame(benchmark_alpha_rows)
benchmark_alpha_selected = select_exploratory_benchmark_alphas(benchmark_alpha_table)
display(benchmark_alpha_selected[[
    "benchmark", "family", "penalty", "alpha",
    "validation_rmse", "n_options_active",
]])
for benchmark in _benchmark_penalties:
    figure = plot_exploratory_alpha_grid(
        benchmark_alpha_table, benchmark_alpha_selected,
        benchmark=benchmark, nominal=prod.notional,
    )
    plt.show()

def selected_benchmark_alpha(benchmark, family, penalty):
    matches = benchmark_alpha_selected.loc[
        benchmark_alpha_selected["benchmark"].eq(benchmark)
        & benchmark_alpha_selected["family"].eq(family)
        & benchmark_alpha_selected["penalty"].eq(penalty)
    ]
    if len(matches) != 1:
        if matches.empty and penalty == "l0":
            return None
        raise ValueError(
            f"Aucun alpha valide et unique pour {benchmark}/{family}/{penalty}."
        )
    return float(matches.iloc[0]["alpha"])


## 1. Réplication de la surface de prix

Les poids sont ajustés sur une grille date–spot, puis évalués hors des points utilisés pour l’ajustement. Le tableau principal compare l’erreur, la parcimonie et la concentration. Les poids complets et les surfaces détaillées sont des diagnostics optionnels.


In [ ]:
price_benchmark_results = {}
price_benchmark_rows = []

for penalty in ["l0", "l1", "l2", "elastic_net", "l1_2"]:
    benchmark_solver = "proximal" if penalty == "l0" else SOLVER
    for family in MAIN_FAMILIES:
        benchmark_alpha = selected_benchmark_alpha("price_surface", family, penalty)
        if benchmark_alpha is None:
            print(f"{FAMILY_LABELS[family]} / L0 : exclu, aucun alpha convergé avec au moins deux options actives.")
            continue
        result = run_naive_benchmark_price(
            product=prod,
            spot0=spot0,
            valuation_date=valuation_date,
            maturity_date=maturity_date,
            rate=rate,
            dividend_yield=dividend_yield,
            vol=vol,
            family=family,
            n_spots=15,
            n_paths=CFG["benchmark_paths"],
            penalty=penalty,
            alpha=benchmark_alpha,
            seed=1042, solver=benchmark_solver,
        )
        price_benchmark_results[(family, penalty)] = result
        price_benchmark_rows.append({
            "family": FAMILY_LABELS[family],
            "penalty": penalty,
            "n_options_active": result["portfolio_metrics"]["n_options_active"],
            "test_rmse": result["metrics_test"].get("rmse", np.nan),
            "test_mae": result["metrics_test"].get("mae", np.nan),
            "max_abs_weight": result["portfolio_metrics"]["max_abs_option_weight"],
            "top_5_concentration": result["portfolio_metrics"]["top_5_option_concentration"],
        })

price_benchmark_table = pd.DataFrame(price_benchmark_rows)
display(price_benchmark_table.sort_values(["penalty", "family"]).reset_index(drop=True))

if SHOW_TECHNICAL_DETAILS:
    for (family, penalty), result in price_benchmark_results.items():
        print(f"=== {FAMILY_LABELS[family]} / {penalty} ===")
        display(result["weights"].head(20))
        display(pd.Series(result["optimization"], name="optimisation"))
        display(format_portfolio_metrics(result["portfolio_metrics"]))
        plot_benchmark_surface(
            result["dates"], result["spots"], result["target_tensor"], result["fitted_tensor"],
            title_prefix=f"[{FAMILY_LABELS[family]} / {penalty}] ",
        )


#### Lecture visuelle : apport des binaires

Le graphique de gauche compare l’erreur hors échantillon. Celui de droite compare le nombre d’options actives. Une amélioration de `full` n’est économiquement intéressante que si la baisse d’erreur compense la complexité et la moindre liquidité introduites par les binaires.


In [ ]:
family_order = [FAMILY_LABELS[family] for family in MAIN_FAMILIES]
penalty_order = [
    penalty for penalty in ["l0", "l1", "l2", "elastic_net", "l1_2"]
    if penalty in set(price_benchmark_table["penalty"])
]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
x = np.arange(len(penalty_order), dtype=float)
width = 0.36

for family_index, family_label in enumerate(family_order):
    family_rows = (
        price_benchmark_table.loc[price_benchmark_table["family"] == family_label]
        .set_index("penalty")
        .reindex(penalty_order)
    )
    offset = (family_index - (len(family_order) - 1) / 2) * width
    axes[0].bar(x + offset, family_rows["test_rmse"], width=width, label=family_label)
    axes[1].bar(x + offset, family_rows["n_options_active"], width=width, label=family_label)

axes[0].set_title("Erreur hors échantillon")
axes[0].set_ylabel("RMSE de test")
axes[1].set_title("Complexité du portefeuille")
axes[1].set_ylabel("Nombre d’options actives")

for axis in axes:
    axis.set_xticks(x)
    axis.set_xticklabels(penalty_order, rotation=20)
    axis.set_xlabel("Pénalité")
    axis.grid(axis="y", alpha=0.25)
    axis.legend()

plt.suptitle("Calls + Puts contre Full : performance et complexité", y=1.03)
plt.tight_layout()
RUN_FIGURES["initial_replication_benchmark"] = fig
plt.show()


## 2. Réplication du payoff moyen

Ici, la cible n’est plus directement le prix courant mais l’espérance risque-neutre du payoff brut. Avant maturité, on utilise la valeur Black-Scholes de chaque option puis on retire l’actualisation pour retrouver cette espérance. Un payoff évalué au spot courant ne conviendrait pas, car le spot terminal est encore inconnu.


In [ ]:
from autocall_replication.replication.mean_payoff import (
    autocall_meanpayoff_on_grid_mc,
    mean_undiscounted_payoff_mc,
    run_naive_benchmark_payoff,
    vanilla_basis_expected_payoff_on_grid,
)


In [ ]:
mean_payoff_results = {}
mean_payoff_rows = []

for family in MAIN_FAMILIES:
    result = run_naive_benchmark_payoff(
        product=prod,
        spot0=spot0,
        valuation_date=valuation_date,
        maturity_date=maturity_date,
        rate=rate,
        dividend_yield=dividend_yield,
        vol=vol,
        family=family,
        n_spots=15,
        n_paths=CFG["benchmark_paths"],
        penalty="l2",
        alpha=selected_benchmark_alpha("mean_payoff", family, "l2"),
        seed=1042, solver=SOLVER,
    )
    mean_payoff_results[family] = result
    mean_payoff_rows.append({
        "family": FAMILY_LABELS[family],
        "train_rmse": result["metrics_train"]["rmse"],
        "test_rmse": result["metrics_test"].get("rmse", np.nan),
        "train_mae": result["metrics_train"]["mae"],
        "test_mae": result["metrics_test"].get("mae", np.nan),
        "n_options_active": result["portfolio_metrics"]["n_options_active"],
    })

mean_payoff_table = pd.DataFrame(mean_payoff_rows)
display(mean_payoff_table)

if SHOW_TECHNICAL_DETAILS:
    for family, result in mean_payoff_results.items():
        display(result["weights"].head(20))
        display(pd.Series(result["optimization"], name=f"optimisation — {FAMILY_LABELS[family]}"))
        display(format_portfolio_metrics(result["portfolio_metrics"]))
        plot_benchmark_surface(
            result["dates"], result["spots"], result["target_tensor"], result["fitted_tensor"],
            title_prefix=f"[Payoff moyen / {FAMILY_LABELS[family]}] ",
        )


## Recherche d'alpha du contrôle Pathwise payoff

Cette recherche L2 reste exploratoire. Pour chaque famille, on calibre les poids
sur un ensemble de trajectoires, puis on choisit sur validation l'alpha donnant
la plus faible RMSE parmi les solutions convergées avec au moins deux options.
Les résultats ne sont pas promus vers le notebook principal.


In [ ]:
from autocall_replication.replication.pathwise_payoff import run_naive_benchmark_payoff_pathwise
from autocall_replication.replication.optimization import SolverConfig, solve_replication
from autocall_replication.replication.hyperparameter_study import (
    HyperparameterGrid, exploratory_alpha_values,
    plot_exploratory_alpha_grid, select_exploratory_benchmark_alphas,
)

pathwise_grid = HyperparameterGrid()
pathwise_alpha_rows = []
for family in MAIN_FAMILIES:
    baseline = run_naive_benchmark_payoff_pathwise(
        product=prod, spot0=spot0, valuation_date=valuation_date,
        maturity_date=maturity_date, rate=rate, dividend_yield=dividend_yield,
        vol=vol, family=family, n_paths=CFG["alpha_paths"],
        penalty="l2", alpha=1e-4, solver="legacy", seed=42,
    )
    split = int(baseline["split"])
    matrix = np.asarray(baseline["X"], dtype=float)
    target = np.asarray(baseline["y"], dtype=float)
    train_A, train_b = matrix[:split], target[:split]
    validation_A, validation_b = matrix[split:], target[split:]
    for alpha in exploratory_alpha_values(pathwise_grid, "l2"):
        fit = solve_replication(
            train_A, train_b,
            SolverConfig(solver="legacy", penalty="l2", alpha=float(alpha)),
        )
        pathwise_alpha_rows.append({
            "benchmark": "pathwise_payoff", "family": family,
            "penalty": "l2", "alpha": float(alpha),
            "converged": bool(fit.converged),
            "validation_rmse": float(np.sqrt(np.mean(
                (validation_b - validation_A @ fit.weights) ** 2
            ))),
            "n_options_active": int(np.count_nonzero(np.abs(fit.weights[1:]) > 1e-6)),
        })

pathwise_alpha_table = pd.DataFrame(pathwise_alpha_rows)
pathwise_alpha_selected = select_exploratory_benchmark_alphas(pathwise_alpha_table)
if len(pathwise_alpha_selected) != len(MAIN_FAMILIES):
    raise ValueError("Il faut un alpha pathwise L2 valide par famille.")
display(pathwise_alpha_selected[[
    "benchmark", "family", "penalty", "alpha",
    "validation_rmse", "n_options_active",
]])
plot_exploratory_alpha_grid(
    pathwise_alpha_table, pathwise_alpha_selected,
    benchmark="pathwise_payoff", nominal=prod.notional,
)
plt.show()

def selected_pathwise_alpha(family):
    matches = pathwise_alpha_selected.loc[
        pathwise_alpha_selected["family"].eq(family)
        & pathwise_alpha_selected["penalty"].eq("l2")
    ]
    if len(matches) != 1:
        raise ValueError(f"Aucun alpha pathwise L2 valide et unique pour {family}.")
    return float(matches.iloc[0]["alpha"])


## 3. Contrôle Pathwise payoff non conditionnel

Une trajectoire fournit ici un unique payoff total actualisé. Le panier est global :
cette expérience ne garantit pas la reproduction des paiements à chaque date.
L'alpha L2 est choisi dans la cellule précédente, uniquement pour ce contrôle
exploratoire ; il n'est ni exporté ni utilisé par la politique conditionnelle.


In [ ]:
from autocall_replication.monte_carlo import autocall_discounted_payoff_from_path
from autocall_replication.replication.pathwise_payoff import (
    build_pathwise_payoff_dataset,
    plot_pathwise_benchmark,
    run_naive_benchmark_payoff_pathwise,
    run_pathwise_penalty_comparison,
    run_pathwise_regularization_path,
    vanilla_discounted_payoff_from_path,
)


In [ ]:
pathwise_results = {}
pathwise_rows = []

for family in MAIN_FAMILIES:
    result = run_naive_benchmark_payoff_pathwise(
        product=prod,
        spot0=spot0,
        valuation_date=valuation_date,
        maturity_date=maturity_date,
        rate=rate,
        dividend_yield=dividend_yield,
        vol=vol,
        family=family,
        n_paths=CFG["pathwise_paths"],
        penalty="l2",
        alpha=selected_pathwise_alpha(family),
        seed=1042, solver=SOLVER,
    )
    pathwise_results[family] = result
    pathwise_rows.append({
        "family": FAMILY_LABELS[family],
        "train_rmse": result["metrics_train"]["rmse"],
        "test_rmse": result["metrics_test"].get("rmse", np.nan),
        "test_mae": result["metrics_test"].get("mae", np.nan),
        "n_options_active": result["portfolio_metrics"]["n_options_active"],
    })

pathwise_benchmark_table = pd.DataFrame(pathwise_rows)
display(pathwise_benchmark_table)

if SHOW_TECHNICAL_DETAILS:
    for family, result in pathwise_results.items():
        display(result["weights"].head(20))
        display(pd.Series(result["optimization"], name=f"optimisation — {FAMILY_LABELS[family]}"))
        display(format_portfolio_metrics(result["portfolio_metrics"]))
        plot_pathwise_benchmark(
            result["y"], result["replica"], title_prefix=f"[Pathwise / {FAMILY_LABELS[family]}] "
        )


## 4. Extensions volontairement exploratoires

### 4.1 Niveau 3 — future couverture dynamique du résidu

La prochaine phase construira une stratégie autofinancée appliquée à $H_0-R_0$. Elle ajoutera les gains du sous-jacent, le compte cash, les rebalancements, la liquidation au rappel et les coûts correspondants :

$$E_{\mathrm{final}} = H_0 - R_0 - G_0^{\mathrm{hedge}}.$$

Le benchmark delta de la section 4.5 couvre l'autocall **seul** ; il ne couvre pas le résidu laissé par la politique B et ne réalise donc pas ce niveau 3. Aucun chiffre de cette section ne doit encore être présenté comme un risque après couverture dynamique du résidu. Le statut reste `level_3_not_implemented`.


### 4.2 Scénarios OTC hypothétiques pour les binaires

Cette extension conserve le payoff binaire exact et ajoute une charge provisoire de
5, 10 ou 25 points de base du payout pour les scénarios favorable, central et stressé.
Ces charges sont des hypothèses de sensibilité, pas des cotations de dealer. Elles ne
figurent donc plus dans le parcours principal.

La cellule est désactivée par défaut. Elle nécessite les politiques conditionnelles,
le prix de l'autocall et la configuration produits par `autocall_research.ipynb` dans
le noyau courant.


In [ ]:
OTC_BINARY_MODES = ("otc_favorable", "otc_central", "otc_stressed")
otc_funding_results = {}
otc_funding_summary = pd.DataFrame()

if not RUN_OTC_FUNDING_EXPLORATION:
    print("Scénarios OTC désactivés. Ils ne font pas partie du parcours principal.")
else:
    _required_context = ("stable_profile_results", "pricing_result", "CFG", "prod", "spot0", "valuation_date", "maturity_date", "rate", "dividend_yield", "vol")
    _missing_context = [name for name in _required_context if name not in globals()]
    if _missing_context:
        raise RuntimeError("Contexte du notebook principal absent : " + ", ".join(_missing_context))
    from autocall_replication.replication.funding import evaluate_conditional_policy_funding
    _full_result = stable_profile_results.get("full")
    if _full_result is None or _full_result["profiles"].empty:
        print("Aucun profil Full stable disponible pour l'exploration OTC.")
    else:
        otc_funding_results["full"] = evaluate_conditional_policy_funding(
            base_result=_full_result, product=prod, spot0=spot0,
            valuation_date=valuation_date, maturity_date=maturity_date,
            rate=rate, dividend_yield=dividend_yield, vol=vol,
            n_paths=CFG["funding_paths"], seed=20500,
            initial_autocall_premium=pricing_result["actualized_price"],
            spread_width=5.0, binary_modes=OTC_BINARY_MODES,
        )
        otc_funding_summary = otc_funding_results["full"]["summary"].copy()
        print("Sensibilité du financement aux trois charges OTC hypothétiques :")
        display(otc_funding_summary)


### 4.3 Coûts bid-ask et données de marché — étape ultérieure optionnelle

Trois objets sont séparés :

1. le **coût par nœud** date–état, calculé à partir des demi-spreads bid-ask pour les calls et puts ;
2. le **coût attendu de la politique**, obtenu en pondérant chaque nœud par sa fréquence d’atteinte ;
3. le **coût OTC hypothétique** des binaires, construit à partir d’un spread vertical synthétique et des scénarios favorable, central et stressé.

Les poids sont continus. La somme non pondérée de tous les nœuds est affichée comme borne haute, mais ne doit pas être confondue avec le coût attendu. Les commissions, arrondis en contrats, marge et financement mark-to-market ne sont pas encore inclus.

Le ratio `coût / réduction d’ES 97,5 %` est un filtre indicatif : il compare un coût certain à une réduction de mesure de risque, pas à un profit certain. Une cotation manquante rend le coût incomplet ; elle n’est jamais remplacée silencieusement par zéro.

Cette analyse est désactivée par défaut et nécessite les politiques calibrées par le
notebook principal dans le noyau courant. Elle reste exploratoire tant que les
cotations, les tailles et la convention de dénouement ne sont pas figées.


In [ ]:
option_snapshot = None
profile_costs = None
cost_risk_filter = None
if not RUN_POLICY_MARKET_COSTS:
    print("Extension bid-ask/OTC désactivée ; aucun coût de marché n'est imputé.")
else:
    _required_context = ("stable_profile_results", "risk_result", "prod", "spot0")
    _missing_context = [name for name in _required_context if name not in globals()]
    if _missing_context:
        raise RuntimeError("Contexte du notebook principal absent : " + ", ".join(_missing_context))
    if yf is None:
        raise RuntimeError("yfinance n'est pas installé : coûts de marché indisponibles.")
    import importlib
    import autocall_replication.replication.transaction_costs as _transaction_costs_module
    _transaction_costs_module = importlib.reload(_transaction_costs_module)
    from autocall_replication.replication.transaction_costs import OTC_COST_SCENARIOS, download_yfinance_option_snapshot, estimate_conditional_policy_transaction_costs
    from autocall_replication.replication.risk import compare_transaction_cost_to_risk_reduction
    MARKET_TICKER = "SPY"
    market_expiries = sorted({pd.Timestamp(instrument.maturity_date).strftime("%Y-%m-%d") for stable_result in stable_profile_results.values() for instrument in stable_result["vanillas"] if instrument.__class__.__name__ != "Cash"})
    try:
        option_snapshot = download_yfinance_option_snapshot(ticker=MARKET_TICKER, expiries=market_expiries)
        conditional_cost_results = {family: estimate_conditional_policy_transaction_costs(base_result=stable_result, snapshot=option_snapshot, autocall_notional=prod.notional, model_spot0=spot0, scenarios=OTC_COST_SCENARIOS) for family, stable_result in stable_profile_results.items() if not stable_result["profiles"].empty}
        profile_costs = {"summary": pd.concat([r["summary"] for r in conditional_cost_results.values()], ignore_index=True), "details": pd.concat([r["details"] for r in conditional_cost_results.values()], ignore_index=True)}
        cost_summary_for_filter = profile_costs["summary"].rename(columns={"expected_transaction_cost": "transaction_cost"})
        cost_risk_filter = compare_transaction_cost_to_risk_reduction(risk_comparison=risk_result["comparison"], cost_summary=cost_summary_for_filter, autocall_notional=prod.notional, risk_metric="es_975_reduction")
        print("Coûts d'exécution exploratoires par profil et scénario OTC :"); display(profile_costs["summary"])
        print("Filtre économique indicatif :"); display(cost_risk_filter)
        if SHOW_TECHNICAL_DETAILS:
            display(pd.DataFrame(OTC_COST_SCENARIOS).T); display(profile_costs["details"])
    except Exception as market_error:
        print("Données de marché indisponibles ou incomplètes :", market_error)
        print("Aucun coût n'est remplacé silencieusement par zéro.")


### 4.4 Exploration de marché optionnelle

Cette cellule est indépendante du contrat synthétique. Elle sert à inspecter une chaîne d’options réelle et la qualité de ses cotations. Elle est désactivée par défaut, car elle dépend du réseau et n’est pas nécessaire à la compréhension du pipeline.

Les variables utilisent le préfixe `market_` afin de ne jamais écraser `prod`, `spot0`, `valuation_date` ou `maturity_date`.


In [ ]:
MARKET_EXPLORATION_TICKER = "SPY"

if not RUN_OPTIONAL_MARKET_EXPLORATION:
    print("Exploration détaillée de marché désactivée. Activer RUN_OPTIONAL_MARKET_EXPLORATION pour l'exécuter.")
elif yf is None:
    print("yfinance n'est pas installé : exploration de marché ignorée.")
else:
    market_ticker_object = yf.Ticker(MARKET_EXPLORATION_TICKER)
    market_spot_history = yf.download(
        MARKET_EXPLORATION_TICKER,
        start="2025-01-01",
        end="2026-06-30",
        interval="1d",
        auto_adjust=True,
        progress=False,
    )
    display(market_spot_history.tail())

    market_spot = market_spot_history["Close"].dropna().iloc[-1].item()
    market_snapshot_date = pd.Timestamp(market_spot_history.index[-1])
    market_expiries_available = market_ticker_object.options

    if not market_expiries_available:
        print("Aucune maturité disponible pour ce ticker.")
    else:
        market_maturity = pd.Timestamp(market_expiries_available[0])
        market_chain = market_ticker_object.option_chain(market_expiries_available[0])
        market_calls = market_chain.calls.copy().assign(kind="call")
        market_puts = market_chain.puts.copy().assign(kind="put")
        market_vanillas = pd.concat([market_calls, market_puts], ignore_index=True)

        market_columns = [
            column for column in
            ["contractSymbol", "kind", "strike", "bid", "ask", "lastPrice", "impliedVolatility", "volume", "openInterest"]
            if column in market_vanillas.columns
        ]
        market_vanillas = market_vanillas[market_columns].copy()

        def _market_bs_price(row):
            sigma = row["impliedVolatility"]
            if pd.isna(sigma) or sigma <= 0:
                return np.nan
            option_class = EuropeanCall if row["kind"] == "call" else EuropeanPut
            market_option = option_class(
                name="market_option",
                strike=float(row["strike"]),
                maturity_date=market_maturity,
                notional=1.0,
            )
            return market_option.price_bs(
                market_spot,
                market_snapshot_date,
                rate,
                dividend_yield,
                sigma,
            )

        market_vanillas["benchmark_price_bs"] = market_vanillas.apply(_market_bs_price, axis=1)
        display(market_vanillas.head(20))
